# UD5.08. Depuración y puesta en servicio

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 16 y 17 de los apuntes · Criterios **2.d** y **2.e**

---

Dos mitades que parecen distintas y son la misma cosa: **que el modelo haga lo que crees
que hace**.

La primera es depurar. En aprendizaje profundo, el código que está mal no da un error: da
un resultado malo, y un resultado malo es indistinguible de un problema difícil si no se
comprueba. Etiquetas desplazadas una posición, normalización aplicada al conjunto entero,
imágenes barajadas sin sus etiquetas, la activación de salida equivocada: nada de eso lanza
una excepción.

La segunda es poner el modelo en servicio, y el fallo característico es el mismo con otra
cara: **el modelo funciona en el cuaderno y da basura en producción**, porque el
preprocesamiento se reescribió y no coincide.

La respuesta a las dos es un método, y el método consiste en **comprobar antes de creer**.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json
import time
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)

CLASES = ["camiseta", "pantalon", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botin"]

(X_todo, y_todo), (X_p, y_p) = keras.datasets.fashion_mnist.load_data()
X_ent, y_ent = X_todo[:8000], y_todo[:8000]
X_val, y_val = X_todo[8000:10000], y_todo[8000:10000]
X_pru, y_pru = X_p, y_p

print(f"entrenamiento {X_ent.shape}   validacion {X_val.shape}   prueba {X_pru.shape}")
print(f"dtype de X: {X_ent.dtype}   rango: [{X_ent.min()}, {X_ent.max()}]")

Fíjate en esa última línea: los datos están **sin normalizar**, en `uint8` de 0 a 255. Es a
propósito: la mitad de este cuaderno consiste en descubrir errores, y ese es uno de los que
se van a descubrir.

---

## 1. Las siete comprobaciones, como función reutilizable

El método del bloque 16.2, empaquetado. Esta función se usa en la P5.3 y en el proyecto
PR5, así que merece la pena escribirla bien una vez.

In [ ]:
def comprueba(construye_modelo, X, y, n_clases, nombres=None, verbose=True):
    # Las siete comprobaciones de sensatez del bloque 16.2, en orden.
    # Devuelve un diccionario con el resultado de cada una. Ninguna tarda
    # mas de unos segundos, y juntas descartan la mayoria de los errores
    # que no dan excepcion.
    informe = {}

    # 1. Mirar los datos. No se automatiza: se dibuja y se mira.
    informe["1_muestras"] = "revisar la figura a mano"

    # 2. Reparto de clases y cota minima.
    cuentas = np.bincount(y.ravel(), minlength=n_clases)
    informe["2_mayoritaria"] = float(cuentas.max() / cuentas.sum())
    informe["2_desequilibrio"] = float(cuentas.max() / max(cuentas.min(), 1))

    # 3. Rango y tipo.
    informe["3_dtype"] = str(X.dtype)
    informe["3_rango"] = (float(X.min()), float(X.max()))
    informe["3_normalizado"] = bool(X.max() <= 1.001 or X.min() >= -1.001 and X.max() <= 1.001)
    informe["3_hay_nan"] = bool(not np.isfinite(X).all())

    # 4. Formas.
    modelo = construye_modelo()
    informe["4_forma_entrada_modelo"] = tuple(modelo.input_shape[1:])
    informe["4_forma_entrada_datos"] = tuple(X.shape[1:])
    informe["4_formas_coinciden"] = (informe["4_forma_entrada_modelo"]
                                     == informe["4_forma_entrada_datos"])
    informe["4_unidades_salida"] = int(modelo.output_shape[-1])
    informe["4_salida_coincide"] = informe["4_unidades_salida"] == n_clases

    # 5. Perdida inicial = log(k).
    esperada = float(np.log(n_clases))
    h = modelo.fit(X[:500], y[:500], epochs=1, batch_size=32, verbose=0)
    real = float(h.history["loss"][0])
    informe["5_perdida_esperada"] = esperada
    informe["5_perdida_real"] = real
    informe["5_desviacion_pct"] = abs(real - esperada) / esperada * 100
    informe["5_pasa"] = informe["5_desviacion_pct"] < 20

    # 6. Sobreajustar 20 muestras. La que casi nadie hace, y la que mas descarta.
    mini = construye_modelo()
    mini.fit(X[:20], y[:20], epochs=200, batch_size=20, verbose=0)
    _, exactitud = mini.evaluate(X[:20], y[:20], verbose=0)
    informe["6_exactitud_20"] = float(exactitud)
    informe["6_pasa"] = exactitud > 0.99

    # 7. El punto de referencia: la clase mayoritaria.
    informe["7_referencia"] = informe["2_mayoritaria"]

    if verbose:
        for clave, valor in informe.items():
            marca = ""
            if clave.endswith("_pasa"):
                marca = "  <- OK" if valor else "  <- FALLA"
            if clave == "3_normalizado" and not valor:
                marca = "  <- FALLA: los datos no estan normalizados"
            if clave == "4_formas_coinciden" and not valor:
                marca = "  <- FALLA"
            print(f"  {clave:26} {str(valor):28}{marca}")
    return informe

In [ ]:
def modelo_v1():
    m = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


print("COMPROBACIONES SOBRE LOS DATOS SIN NORMALIZAR:")
informe_v1 = comprueba(modelo_v1, X_ent, y_ent, 10)

La comprobación 3 y la 5 lo han pillado: los datos van de 0 a 255 y la pérdida inicial está
lejísimos de $\log(10) = 2{,}303$. Con entradas de ese tamaño, la primera capa produce
activaciones enormes y el modelo arranca desde un sitio del que le cuesta mucho salir.

Arreglémoslo y volvamos a comprobar.

In [ ]:
X_ent_n = X_ent.astype("float32") / 255.0
X_val_n = X_val.astype("float32") / 255.0
X_pru_n = X_pru.astype("float32") / 255.0

print("COMPROBACIONES SOBRE LOS DATOS NORMALIZADOS:")
informe_v2 = comprueba(modelo_v1, X_ent_n, y_ent, 10)

In [ ]:
# Lo que cuesta el error: las dos versiones, mismo modelo, mismas epocas.
resultados = []
for etiqueta, datos in [("sin normalizar", (X_ent, X_val, X_pru)),
                        ("normalizados", (X_ent_n, X_val_n, X_pru_n))]:
    keras.utils.set_random_seed(SEMILLA)
    m = modelo_v1()
    m.fit(datos[0], y_ent, epochs=10, batch_size=64,
          validation_data=(datos[1], y_val), verbose=0)
    _, acc = m.evaluate(datos[2], y_pru, verbose=0)
    resultados.append((etiqueta, acc))
    print(f"{etiqueta:16} exactitud de prueba {acc:.4f}")

print()
print(f"Diferencia: {resultados[1][1] - resultados[0][1]:+.4f}")
print()
print("Y la version sin normalizar NO da ningun error. Entrena, converge, y da")
print("una cifra que parece razonable si no tienes la otra al lado. Esa es la")
print("forma en la que fallan los errores de esta unidad.")

### La comprobación seis, en detalle

Es la que más descarta y la que casi nadie hace. Se cogen veinte muestras, se entrena sobre
ellas sin validación ni regularización, y se comprueba que el modelo las **memoriza por
completo**.

Un modelo que no puede memorizar veinte ejemplos tiene un fallo de implementación, y
ninguna cantidad de épocas sobre el conjunto entero lo va a arreglar.

In [ ]:
def modelo_roto_salida():
    # El error: la ultima capa no tiene activacion, y la perdida espera
    # probabilidades. No da ninguna excepcion.
    m = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10),                      # <-- sin softmax
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


print("El modelo con la salida rota, sobre 20 muestras:")
for nombre, construye in [("correcto", modelo_v1), ("sin softmax", modelo_roto_salida)]:
    keras.utils.set_random_seed(SEMILLA)
    m = construye()
    m.fit(X_ent_n[:20], y_ent[:20], epochs=200, batch_size=20, verbose=0)
    _, exactitud = m.evaluate(X_ent_n[:20], y_ent[:20], verbose=0)
    print(f"  {nombre:14} exactitud sobre las 20: {exactitud:.3f}")
print()
print("En este caso el modelo roto TAMBIEN memoriza, porque con logits y")
print("sparse_categorical_crossentropy Keras hace algo razonable aunque no sea")
print("lo que pediste. La comprobacion 6 no lo pilla, y la 5 si: mira la")
print("perdida inicial de los dos.")

In [ ]:
for nombre, construye in [("correcto", modelo_v1), ("sin softmax", modelo_roto_salida)]:
    keras.utils.set_random_seed(SEMILLA)
    m = construye()
    h = m.fit(X_ent_n[:500], y_ent[:500], epochs=1, batch_size=32, verbose=0)
    print(f"  {nombre:14} perdida inicial {h.history['loss'][0]:.3f}   "
          f"esperada {np.log(10):.3f}")
print()
print("Ninguna comprobacion las pilla todas. Por eso se hacen las siete, en orden,")
print("y por eso se para en la primera que falla en vez de seguir hasta el final.")

---

## 2. Los errores que no dan error

Cuatro de los más comunes, fabricados y medidos. Para cada uno: el síntoma, la comprobación
que lo pilla, y lo que cuesta.

In [ ]:
def entrena_rapido(X, y, Xv, yv, Xp, yp, construye=None, epocas=8):
    keras.utils.set_random_seed(SEMILLA)
    m = (construye or modelo_v1)()
    m.fit(X, y, epochs=epocas, batch_size=64, validation_data=(Xv, yv), verbose=0)
    _, acc = m.evaluate(Xp, yp, verbose=0)
    return m, acc


_, acc_bien = entrena_rapido(X_ent_n, y_ent, X_val_n, y_val, X_pru_n, y_pru)
print(f"referencia (todo bien): {acc_bien:.4f}")

In [ ]:
casos = {}

# ERROR A: las etiquetas desplazadas una posicion.
_, casos["etiquetas desplazadas"] = entrena_rapido(
    X_ent_n, np.roll(y_ent, 1), X_val_n, y_val, X_pru_n, y_pru)

# ERROR B: X e y barajados por separado.
rng = np.random.default_rng(SEMILLA)
_, casos["X e y desalineados"] = entrena_rapido(
    X_ent_n[rng.permutation(len(X_ent_n))], y_ent, X_val_n, y_val, X_pru_n, y_pru)

# ERROR C: normalizar con una constante distinta en entrenamiento y en prueba.
_, casos["normalizacion incoherente"] = entrena_rapido(
    X_ent_n, y_ent, X_val_n, y_val, X_pru.astype("float32") / 128.0, y_pru)

# ERROR D: normalizar dos veces.
_, casos["normalizado dos veces"] = entrena_rapido(
    X_ent_n / 255.0, y_ent, X_val_n / 255.0, y_val, X_pru_n / 255.0, y_pru)

print(f"{'error':30} {'exactitud':>10} {'caida':>8}")
print("-" * 52)
print(f"{'(ninguno)':30} {acc_bien:>10.4f} {0.0:>8.4f}")
for nombre, acc in casos.items():
    print(f"{nombre:30} {acc:>10.4f} {acc - acc_bien:>8.4f}")

### Cuál pilla cada comprobación

| Error | Comprobación que lo pilla | Cómo |
|---|---|---|
| Etiquetas desplazadas | **1**, mirar los datos | la rejilla enseña un pantalón etiquetado como camiseta |
| `X` e `y` desalineados | **1** y **6** | la rejilla no cuadra, y no memoriza 20 muestras |
| Normalización incoherente | **3** | el rango de prueba no coincide con el de entrenamiento |
| Normalizado dos veces | **3** y **5** | el rango es $[0, 0{,}004]$ y la pérdida inicial se desvía |

El primero y el segundo se parecen en la cifra y **no se parecen nada en el síntoma**: con
las etiquetas desplazadas el modelo aprende una permutación consistente y puede llegar a
memorizar el entrenamiento; con `X` e `y` desalineados no hay nada que aprender. Mirar la
curva de entrenamiento los distingue.

In [ ]:
# La comprobacion 3, escrita como una funcion que se puede llamar siempre.
def comprueba_coherencia(X_entrena, X_prueba, nombre_a="entrenamiento",
                         nombre_b="prueba", tolerancia=0.1):
    a = (float(X_entrena.min()), float(X_entrena.max()),
         float(X_entrena.mean()), float(X_entrena.std()))
    b = (float(X_prueba.min()), float(X_prueba.max()),
         float(X_prueba.mean()), float(X_prueba.std()))
    print(f"{'':12} {'min':>9} {'max':>9} {'media':>9} {'desv':>9}")
    print(f"{nombre_a:12} {a[0]:>9.4f} {a[1]:>9.4f} {a[2]:>9.4f} {a[3]:>9.4f}")
    print(f"{nombre_b:12} {b[0]:>9.4f} {b[1]:>9.4f} {b[2]:>9.4f} {b[3]:>9.4f}")
    desviacion = abs(a[2] - b[2]) / max(abs(a[2]), 1e-9)
    print(f"\ndesviacion relativa de la media: {desviacion:.1%}", end="  ")
    print("OK" if desviacion < tolerancia else "<- SOSPECHOSO")
    return desviacion < tolerancia


print("Correcto:")
comprueba_coherencia(X_ent_n, X_pru_n)
print()
print("Con la normalizacion incoherente:")
comprueba_coherencia(X_ent_n, X_pru.astype("float32") / 128.0)

---

## 3. Lo más peligroso: un resultado demasiado bueno

> **Una exactitud del 99 % en un problema difícil no es una buena noticia: es un síntoma.**

Lo primero que hay que sospechar es una **fuga de información**: que una columna de `X`
contenga información del futuro, o de la propia etiqueta. Es exactamente lo que la UD3
enseñó a evitar con la fecha de corte.

Vamos a fabricarla, para saber reconocerla.

In [ ]:
# La fuga: una columna que es casi la etiqueta, colada entre las demas.
X_con_fuga = X_ent_n.reshape(len(X_ent_n), -1).copy()
ruido = np.random.default_rng(SEMILLA).normal(0, 0.05, len(X_con_fuga))
X_con_fuga = np.column_stack([X_con_fuga, y_ent / 10.0 + ruido])

X_pru_fuga = np.column_stack([X_pru_n.reshape(len(X_pru_n), -1),
                              y_pru / 10.0 + np.random.default_rng(1).normal(
                                  0, 0.05, len(X_pru_n))])

keras.utils.set_random_seed(SEMILLA)
con_fuga = keras.Sequential([
    keras.layers.Input(shape=(785,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
con_fuga.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
con_fuga.fit(X_con_fuga, y_ent, epochs=8, batch_size=64, verbose=0)
_, acc_fuga = con_fuga.evaluate(X_pru_fuga, y_pru, verbose=0)

print(f"exactitud sin fuga: {acc_bien:.4f}")
print(f"exactitud CON fuga: {acc_fuga:.4f}")
print()
print("Y las curvas son preciosas, y la validacion acompaña, y la matriz de")
print("confusion es una diagonal perfecta. Ninguna de las siete comprobaciones")
print("lo detecta: todas miran si el modelo funciona, y este funciona.")

### Cómo se detecta una fuga

Ninguna comprobación automática la pilla, porque técnicamente todo está bien. Lo que la
pilla es una lista de preguntas:

| Pregunta | Por qué |
|---|---|
| ¿Alguna característica se calcula **después** del momento en que se predice? | es la fuga clásica, la de la fecha de corte de la UD3 |
| ¿Alguna característica es una función de la etiqueta? | fabricada, o colada por el proceso de extracción |
| ¿Se normalizó, seleccionaron variables o rellenaron ausentes **antes** de partir? | la partición de prueba habrá participado |
| ¿Hay duplicados entre entrenamiento y prueba? | el modelo los ha visto |
| ¿El resultado es mejor de lo que un experto humano conseguiría? | si lo es, hay que explicarlo |

Y una comprobación concreta que sí se puede automatizar: **la importancia de cada
característica**. Si una sola domina de forma absurda, mírala.

In [ ]:
# Una medida barata de importancia: permutar una columna y ver cuanto empeora.
def importancia_por_permutacion(modelo, X, y, columnas, rng, n=8):
    _, base = modelo.evaluate(X, y, verbose=0)
    caidas = []
    for col in columnas:
        X_p = X.copy()
        X_p[:, col] = rng.permutation(X_p[:, col])
        _, con = modelo.evaluate(X_p, y, verbose=0)
        caidas.append((col, base - con))
    return sorted(caidas, key=lambda t: -t[1])[:n]


rng = np.random.default_rng(SEMILLA)
columnas = list(rng.choice(784, 20, replace=False)) + [784]   # 784 es la fuga
top = importancia_por_permutacion(con_fuga, X_pru_fuga[:2000], y_pru[:2000],
                                  columnas, rng)

print(f"{'columna':>9} {'caida de exactitud al permutarla':>34}")
print("-" * 46)
for col, caida in top:
    marca = "   <- la columna sospechosa" if col == 784 else ""
    print(f"{col:>9} {caida:>34.4f}{marca}")
print()
print("Una sola columna que se lleva toda la exactitud es una señal de alarma.")
print("No demuestra que haya fuga, pero dice exactamente donde mirar.")

---

## 4. Del modelo al servicio

Ahora la segunda mitad. Un modelo entrenado no es una aplicación: para que lo sea hace
falta que **otra persona pueda usarlo sin abrir el cuaderno**.

In [ ]:
# El modelo que se va a poner en servicio, con el preprocesamiento DENTRO.
keras.utils.set_random_seed(SEMILLA)
servicio = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Rescaling(1.0 / 255),        # <-- dentro del modelo, y viaja con el
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation="softmax"),
], name="clasificador_prendas")
servicio.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Ojo: ahora se le pasan los datos CRUDOS, porque escala el modelo.
servicio.fit(X_ent, y_ent, epochs=12, batch_size=64,
             validation_data=(X_val, y_val), verbose=0,
             callbacks=[keras.callbacks.EarlyStopping(
                 monitor="val_accuracy", mode="max", patience=4,
                 restore_best_weights=True)])
_, acc_servicio = servicio.evaluate(X_pru, y_pru, verbose=0)
print(f"exactitud de prueba: {acc_servicio:.4f}")

### 4.1. Los formatos de guardado

| Formato | Qué guarda | Cuándo |
|---|---|---|
| `.keras` | todo: arquitectura, pesos, estado del optimizador | **por defecto, siempre** |
| `.weights.h5` | solo los pesos | cuando el código de la arquitectura ya se tiene |
| SavedModel (`export`) | formato de TensorFlow Serving | para desplegar fuera de Python |

In [ ]:
carpeta = tempfile.mkdtemp()
filas = []

ruta_keras = os.path.join(carpeta, "modelo.keras")
t0 = time.perf_counter();  servicio.save(ruta_keras);  t_guardar = time.perf_counter() - t0
t0 = time.perf_counter();  recuperado = keras.models.load_model(ruta_keras)
t_cargar = time.perf_counter() - t0
filas.append({"formato": ".keras", "MB": os.path.getsize(ruta_keras) / 1e6,
              "ms guardar": t_guardar * 1000, "ms cargar": t_cargar * 1000,
              "sigue entrenando": True})

ruta_pesos = os.path.join(carpeta, "modelo.weights.h5")
t0 = time.perf_counter();  servicio.save_weights(ruta_pesos)
t_guardar = time.perf_counter() - t0
filas.append({"formato": ".weights.h5", "MB": os.path.getsize(ruta_pesos) / 1e6,
              "ms guardar": t_guardar * 1000, "ms cargar": np.nan,
              "sigue entrenando": False})

print(pd.DataFrame(filas).to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()

# Y la comprobacion obligatoria: el modelo cargado predice lo MISMO.
a = servicio.predict(X_pru[:200], verbose=0)
b = recuperado.predict(X_pru[:200], verbose=0)
print(f"diferencia maxima entre el original y el cargado: {np.abs(a - b).max():.2e}")
assert np.allclose(a, b, atol=1e-6)
print("Identicos. Si no lo fueran, el modelo desplegado no seria el que mediste.")

### 4.2. Lo que el `.keras` NO guarda

Y sin lo cual el fichero da números sin significado:

- los **nombres de las clases**,
- el **umbral** de decisión, si lo hay,
- la media y la desviación de la normalización, **si está fuera del modelo**,
- el **orden exacto** de las columnas de `X`, en datos tabulares,
- y la ficha del modelo: para qué sirve y para qué no.

In [ ]:
preprocesado = {
    "nombre": "clasificador_prendas",
    "version": "1.0",
    "entrenado": "2026-09-11",
    "forma_entrada": [28, 28],
    "dtype_entrada": "uint8, de 0 a 255",
    "escalado": "dentro del modelo, capa Rescaling(1/255)",
    "clases": CLASES,
    "confianza_minima": 0.60,
    "exactitud_prueba": round(float(acc_servicio), 4),
    "poblacion_medida": "las 10.000 imagenes de prueba de Fashion-MNIST",
}

ruta_json = os.path.join(carpeta, "preprocesado.json")
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(preprocesado, f, ensure_ascii=False, indent=2)

print(json.dumps(preprocesado, ensure_ascii=False, indent=2))

### 4.3. La función de inferencia

Recibe un dato crudo y devuelve una **decisión**, no un vector de números. Es lo que va en
`inferencia.py` del proyecto PR5.

In [ ]:
def carga_servicio(ruta_modelo, ruta_json):
    modelo = keras.models.load_model(ruta_modelo)
    with open(ruta_json, encoding="utf-8") as f:
        config = json.load(f)
    return modelo, config


def predice(modelo, config, imagen):
    # imagen: array (28, 28) en uint8, tal como sale de una camara o un fichero.
    if imagen.shape != tuple(config["forma_entrada"]):
        raise ValueError(f"esperaba {config['forma_entrada']}, recibi {imagen.shape}")

    p = modelo.predict(imagen[np.newaxis], verbose=0)[0]
    i = int(p.argmax())
    confianza = float(p[i])

    # Lo que hace un softmax con algo que no es de ninguna clase: contestar una
    # de las k, con confianza alta. La aplicacion tiene que preverlo.
    if confianza < config["confianza_minima"]:
        return {"clase": None, "confianza": confianza,
                "mensaje": "no reconocida con confianza suficiente",
                "mejor_candidata": config["clases"][i]}
    return {"clase": config["clases"][i], "confianza": confianza}


modelo_s, config_s = carga_servicio(ruta_keras, ruta_json)

for idx in (0, 1, 2):
    print(f"real: {CLASES[y_pru[idx]]:12} -> {predice(modelo_s, config_s, X_pru[idx])}")

In [ ]:
# Y el caso que hay que prever: algo que no es ninguna de las diez clases.
rng = np.random.default_rng(SEMILLA)
basura = {
    "ruido puro": rng.integers(0, 256, (28, 28), dtype="uint8"),
    "todo negro": np.zeros((28, 28), dtype="uint8"),
    "todo blanco": np.full((28, 28), 255, dtype="uint8"),
}

for nombre, imagen in basura.items():
    r = predice(modelo_s, config_s, imagen)
    print(f"{nombre:14} -> {r}")
print()
print("Fijate en la confianza de algunas: un softmax NO sabe decir 'no lo se'.")
print("Reparte la probabilidad entre las diez clases que conoce, y si una")
print("gana por poco sale igualmente con un numero que parece una probabilidad.")
print("Ese umbral de confianza minima es la unica defensa que tiene el servicio,")
print("y hay que elegirlo mirando el reparto de confianzas, no a ojo.")

In [ ]:
# Como se elige ese umbral: mirando el reparto de confianzas de los aciertos
# y de los fallos. Es el mismo razonamiento del punto de trabajo de la UD4.
p = modelo_s.predict(X_pru, verbose=0)
confianza = p.max(axis=1)
acierta = p.argmax(axis=1) == y_pru

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
ejes[0].hist(confianza[acierta], bins=40, alpha=0.7, label="aciertos", density=True)
ejes[0].hist(confianza[~acierta], bins=40, alpha=0.7, label="fallos", density=True)
ejes[0].axvline(config_s["confianza_minima"], color="crimson", ls="--",
                label=f"umbral {config_s['confianza_minima']}")
ejes[0].set_xlabel("confianza del modelo");  ejes[0].set_ylabel("densidad")
ejes[0].set_title("Los fallos son menos confiados,\npero se solapan mucho con los aciertos")
ejes[0].legend(fontsize=8)

umbrales = np.linspace(0.2, 0.99, 60)
cobertura = [(confianza >= u).mean() for u in umbrales]
exactitud = [acierta[confianza >= u].mean() if (confianza >= u).any() else np.nan
             for u in umbrales]
ejes[1].plot(cobertura, exactitud, "o-", ms=3)
ejes[1].set_xlabel("cobertura: proporcion de casos que el servicio contesta")
ejes[1].set_ylabel("exactitud sobre los contestados")
ejes[1].set_title("Contestar menos casos permite acertar mas.\nEl umbral elige el punto")
fig.tight_layout()
plt.show()

for u in (0.5, 0.6, 0.8, 0.95):
    sel = confianza >= u
    print(f"umbral {u:.2f}  contesta el {sel.mean():>5.1%} de los casos  "
          f"y acierta el {acierta[sel].mean():.1%}")

> Esa figura es la del criterio 2.e aplicada al servicio: **rechazar los casos dudosos sube
> la exactitud de los que se contestan**, y cuánto se puede rechazar es una decisión de
> negocio, no técnica. Es exactamente el punto de trabajo de la P4.2, con otra pareja de
> cantidades.

---

## 5. Qué cuesta predecir

In [ ]:
una = X_pru[:1]
modelo_s.predict(una, verbose=0)          # calentar

tiempos = []
for _ in range(50):
    t0 = time.perf_counter()
    modelo_s.predict(una, verbose=0)
    tiempos.append(time.perf_counter() - t0)
latencia = np.median(tiempos)

filas = []
for lote in (1, 8, 32, 128, 512, 2048):
    X_lote = X_pru[:lote]
    modelo_s.predict(X_lote, batch_size=lote, verbose=0)
    t0 = time.perf_counter()
    for _ in range(3):
        modelo_s.predict(X_lote, batch_size=lote, verbose=0)
    total = (time.perf_counter() - t0) / 3
    filas.append({"lote": lote, "ms totales": total * 1000,
                  "ms por muestra": total / lote * 1000,
                  "muestras/s": lote / total})

rendimiento = pd.DataFrame(filas)
print(rendimiento.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"Latencia con lote de 1: {latencia * 1000:.1f} ms")
print(f"Ganancia del lote de 2048 frente al de 1, por muestra: "
      f"{rendimiento['ms por muestra'].iloc[0] / rendimiento['ms por muestra'].iloc[-1]:.0f}x")
print()
print("Es la misma idea que la vectorizacion de la UD3: el coste fijo por")
print("llamada se reparte entre las muestras del lote. Por eso los servicios")
print("reales agrupan peticiones, a cambio de añadir algo de latencia.")

In [ ]:
fig, eje = plt.subplots(figsize=(7, 4.2))
eje.plot(rendimiento["lote"], rendimiento["muestras/s"], "o-")
eje.set_xscale("log", base=2)
eje.set_yscale("log")
eje.set_xlabel("tamaño de lote")
eje.set_ylabel("muestras por segundo")
eje.set_title("Agrupar peticiones multiplica el rendimiento,\n"
              "y se aplana cuando el coste fijo deja de dominar")
fig.tight_layout()
plt.show()

---

## 6. La ficha del modelo

Cierra la unidad y enlaza con la UD4: **un límite declarado vale más que una décima de
exactitud**.

In [ ]:
from sklearn.metrics import classification_report

pred = modelo_s.predict(X_pru, verbose=0).argmax(axis=1)
informe_clases = pd.DataFrame(classification_report(
    y_pru, pred, target_names=CLASES, output_dict=True, zero_division=0)).T
peores = informe_clases.loc[CLASES].sort_values("f1-score").head(3)

peor_nombres = ", ".join(peores.index)
peor_f1 = ", ".join(f"{v:.2f}" for v in peores["f1-score"])

ficha = [
    "# Ficha del modelo: clasificador_prendas 1.0",
    "",
    "| Apartado | |",
    "|---|---|",
    "| Para que sirve | Clasificar una imagen de 28x28 en gris en una de diez "
    "categorias de ropa |",
    "| Para que NO sirve | Imagenes en color, con fondo, de prendas fuera de las "
    "diez, o de cualquier otra cosa |",
    f"| Datos de entrenamiento | Fashion-MNIST, {len(X_ent):,} imagenes, "
    "conjunto publico de 2017 |",
    f"| Poblacion de medida | Las {len(X_pru):,} imagenes de prueba de Fashion-MNIST |",
    "| Reparto de clases | equilibrado, 10 % por clase |",
    f"| Exactitud | {acc_servicio:.3f} |",
    "| Punto de referencia | clase mayoritaria: 0,100 |",
    f"| Donde va peor | {peor_nombres} (F1 {peor_f1}) |",
    "| Entradas fuera de alcance | Se rechazan por debajo de una confianza de "
    f"{config_s['confianza_minima']} |",
    f"| Latencia | {latencia * 1000:.0f} ms por peticion suelta, en CPU |",
    f"| Tamaño | {os.path.getsize(ruta_keras) / 1e6:.1f} MB |",
    "| Limite conocido | Las prendas de torso (camiseta, camisa, jersey, abrigo) "
    "se confunden entre si, y es un limite del conjunto de datos mas que del modelo |",
]
print("\n".join(ficha))

---

## Ejercicios

### Ejercicio 1. Amplía la función

Añade a `comprueba()` dos comprobaciones más:

- que no haya **duplicados** entre entrenamiento y prueba,
- que la partición de validación tenga **el mismo reparto de clases** que la de
  entrenamiento, con una tolerancia que tú fijes.

Pruébalas fabricando los dos problemas.

### Ejercicio 2. El error que no está en la lista

Fabrica un quinto error de los que no dan error, distinto de los cuatro del apartado 2, y
documenta el síntoma, la comprobación que lo pilla y lo que cuesta. Algunas ideas:
mezclar las particiones de validación y prueba, aplicar aumento de datos también en la
validación, o entrenar con una métrica que no corresponde a la pérdida.

### Ejercicio 3. La fuga en TechStore

Coge `clientes_abandono.csv` y añade una columna calculada con la ventana de resultado.
Entrena, mide, y aplica la importancia por permutación del apartado 3 para detectarla.
¿Cuánto sube el AUC, y en qué puesto sale la columna culpable?

### Ejercicio 4. El umbral del servicio

Elige el umbral de confianza mínima con un criterio de negocio: un fallo cuesta 5 euros, un
caso rechazado que hay que revisar a mano cuesta 1. Calcula el umbral que minimiza el coste
total y compáralo con el 0,60 de este cuaderno. Dibuja el coste frente al umbral.

### Ejercicio 5. `inferencia.py`

Convierte el apartado 4.3 en un `inferencia.py` de verdad, que se ejecute desde la terminal
con `python inferencia.py imagen.png` e imprima la clase y la confianza. Pruébalo con una
imagen guardada en disco, no con un array. Es el entregable 3.3 del proyecto PR5.

### Ejercicio 6. El preprocesamiento fuera

Entrena el mismo modelo con la normalización **fuera**, guárdalo, y escribe un
`inferencia_mal.py` que se olvide de dividir entre 255. Mide la exactitud de los dos
servicios sobre las mismas imágenes. Escribe tres frases sobre por qué meter el
preprocesamiento dentro del modelo no es una comodidad: es una medida de seguridad.

---

## Lo que hay que llevarse de aquí

1. **En aprendizaje profundo, el código que está mal no da un error: da un resultado
   malo.**
2. **Las siete comprobaciones se hacen en orden y se para en la primera que falla.**
   Ninguna las pilla todas.
3. **Sobreajustar veinte muestras** descarta la mitad de los problemas posibles en cinco
   segundos.
4. **La pérdida inicial tiene que valer $\log(k)$**, y pilla los errores que la
   comprobación 6 deja pasar.
5. **Un resultado demasiado bueno es una fuga de información** hasta que se demuestre lo
   contrario, y ninguna comprobación automática la detecta: la detecta una lista de
   preguntas.
6. **La importancia por permutación dice dónde mirar** cuando se sospecha una fuga.
7. **El preprocesamiento va dentro del modelo**, y así viaja con él.
8. **Un `.keras` no guarda los nombres de las clases ni el umbral**, y sin ellos da números
   sin significado.
9. **Un softmax no sabe decir "no lo sé"**, y el servicio tiene que preverlo con un umbral
   de confianza elegido mirando el reparto, no a ojo.
10. **Rechazar casos dudosos sube la exactitud de los contestados**, y cuánto rechazar es
    una decisión de negocio.
11. **Agrupar peticiones multiplica el rendimiento**, y es la vectorización de la UD3 otra
    vez.
12. **Lo que el modelo no sabe hacer se escribe.**